<a href="https://colab.research.google.com/github/SamurAIGPT/Text-To-Video-AI/blob/main/Text_to_Video_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Steps for generating video from text

1. Use openai to generate a video script from a topic
2. Use edgetts to pick a voice and create a audio based on the above generated script
3. Use whisper and get timed captions for the above audio
4. Now generate visual keywords for the video script using openai api
5. Fetch videos based on the above visual keywords using pexels api
6. Stich together the videos, audio and captions using Moviepy

### Install python 3.11

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y python3.11
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --config python3
!sudo apt-get install -y python3.11-distutils
!wget https://bootstrap.pypa.io/get-pip.py
!python3.11 get-pip.py

In [ ]:
!git clone https://github.com/SamurAIGPT/Text-To-Video-AI

In [ ]:
%cd Text-To-Video-AI

### Install dependencies

In [ ]:
!pip3.11 install -r requirements.txt

### Setup api keys

In [ ]:
import os

# Выберите провайдера LLM: openai | groq | gemini
os.environ["LLM_PROVIDER"] = "openai"

# Укажите ключи API в соответствии с выбранным провайдером
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

# Ключ Pexels API (требуется только если вы используете stock_video или stock_keywords)
os.environ["PEXELS_API_KEY"] = "your_pexels_api_key_here"


### 3. Предварительная загрузка моделей (Опционально)
Эта ячейка загрузит и кэширует локальные нейросетевые модели озвучки Silero и генерации изображений SDXL Turbo. Это позволит контролировать прогресс скачивания.

In [ ]:
# 1. Предварительная загрузка локальной модели озвучки Silero TTS (для русского языка)
import torch
print("Загрузка модели Silero TTS...")
model, example_text = torch.hub.load(
    repo_or_dir='snakers4/silero-models',
    model='silero_tts',
    language='ru',
    speaker='v4_ru'
)
print("Модель Silero TTS успешно загружена!")

# 2. Предварительная загрузка модели SDXL Turbo для локального бэкенда картинок (раскомментируйте при необходимости)
# !pip install diffusers transformers accelerate
# from diffusers import AutoPipelineForText2Image
# print("Загрузка модели SDXL Turbo...")
# pipeline = AutoPipelineForText2Image.from_pretrained(
#     "stabilityai/sdxl-turbo", 
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
# )
# print("Модель SDXL Turbo успешно загружена!")


### Modify permissions for Imagemagick (workaround in colab)

In [ ]:
!apt install imagemagick &> /dev/null
!sed -i '/<policy domain="path" rights="none" pattern="@\*"/d' /etc/ImageMagick-6/policy.xml

### Generate video from topic

In [ ]:
# Запуск пайплайна с темой. 
# Флаг --backend none сгенерирует озвучку на русском и видео с субтитрами на черном фоне (оффлайн режим).
# Вы можете изменить backend на stock_video для загрузки B-Roll видео из Pexels.
!python app.py "Интересные факты о космосе" --backend none


### Display video

In [ ]:
from IPython.display import HTML
from base64 import b64encode

# Path to the video file
video_path = 'rendered_video.mp4'

# Function to display video
def display_video(video_path, width=640, height=480):
    # Load video
    video_file = open(video_path, "rb").read()
    video_url = "data:video/mp4;base64," + b64encode(video_file).decode()
    return HTML(f"""
    <video width={width} height={height} controls>
        <source src="{video_url}" type="video/mp4">
    </video>
    """)

# Display video
display_video(video_path)